# Fraud Detection Project Notebook

This version fixes the Windows path unicode escape error.

In [ ]:
# !pip install pandas numpy scikit-learn xgboost lightgbm

In [1]:
r"""
Project-ready fraud / anomaly detection notebook script backend.

Included models
- Isolation Forest
- Local Outlier Factor (LOF)
- Gradient Boosting baseline:
    * XGBoost if installed
    * else LightGBM if installed
    * else HistGradientBoostingClassifier from scikit-learn
- Optional One-Class SVM
- Optional Elliptic Envelope

Pre-filled for your local Windows folder:
C:\Users\alexa\Downloads\kaggle\

Main IEEE-CIS files expected:
- train_transaction.csv
- train_identity.csv
- test_transaction.csv
- test_identity.csv

Other CSVs in the same folder can also be tested with the generic pipeline.
"""

from __future__ import annotations

import json
import warnings
from pathlib import Path
from typing import Iterable, Optional, Tuple, Dict

import numpy as np
import pandas as pd

from sklearn.compose import ColumnTransformer
from sklearn.covariance import EllipticEnvelope
from sklearn.ensemble import HistGradientBoostingClassifier, IsolationForest
from sklearn.impute import SimpleImputer
from sklearn.metrics import (
    average_precision_score,
    roc_auc_score,
    precision_recall_curve,
    precision_score,
    recall_score,
    f1_score,
    confusion_matrix,
)
from sklearn.model_selection import train_test_split
from sklearn.neighbors import LocalOutlierFactor
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import OrdinalEncoder, StandardScaler
from sklearn.svm import OneClassSVM

warnings.filterwarnings("ignore")

# =========================
# PRE-FILLED PATHS
# =========================
BASE_DIR = Path(r"C:\Users\alexa\Downloads\kaggle")
IEEE_TRAIN_TRANSACTION = BASE_DIR / "train_transaction.csv"
IEEE_TRAIN_IDENTITY = BASE_DIR / "train_identity.csv"
IEEE_TEST_TRANSACTION = BASE_DIR / "test_transaction.csv"
IEEE_TEST_IDENTITY = BASE_DIR / "test_identity.csv"

IBM_CREDIT_CSV = BASE_DIR / "credit_card_transactions-ibm_v2.csv"
USER0_CREDIT_CSV = BASE_DIR / "User0_credit_card_transactions.csv"
SD254_CARDS_CSV = BASE_DIR / "sd254_cards.csv"
SD254_USERS_CSV = BASE_DIR / "sd254_users.csv"

OUTPUT_DIR = BASE_DIR / "project_outputs"
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

COMMON_LABEL_CANDIDATES = [
    "isfraud", "fraud", "class", "label", "target", "is_fraud",
    "fraud_flag", "bad_flag", "default", "y"
]
COMMON_TIME_CANDIDATES = [
    "transactiondt", "timestamp", "datetime", "date", "time", "event_time", "ts"
]


def lower_map(columns: Iterable[str]) -> Dict[str, str]:
    return {c.lower(): c for c in columns}


def detect_label_column(df: pd.DataFrame) -> Optional[str]:
    cmap = lower_map(df.columns)
    for c in COMMON_LABEL_CANDIDATES:
        if c in cmap:
            return cmap[c]
    return None


def detect_time_column(df: pd.DataFrame) -> Optional[str]:
    cmap = lower_map(df.columns)
    for c in COMMON_TIME_CANDIDATES:
        if c in cmap:
            return cmap[c]
    return None


def reduce_memory_usage(df: pd.DataFrame) -> pd.DataFrame:
    for col in df.columns:
        dtype = df[col].dtype
        if pd.api.types.is_integer_dtype(dtype):
            c_min, c_max = df[col].min(), df[col].max()
            if pd.isna(c_min) or pd.isna(c_max):
                continue
            if c_min >= np.iinfo(np.int8).min and c_max <= np.iinfo(np.int8).max:
                df[col] = df[col].astype(np.int8)
            elif c_min >= np.iinfo(np.int16).min and c_max <= np.iinfo(np.int16).max:
                df[col] = df[col].astype(np.int16)
            elif c_min >= np.iinfo(np.int32).min and c_max <= np.iinfo(np.int32).max:
                df[col] = df[col].astype(np.int32)
        elif pd.api.types.is_float_dtype(dtype):
            df[col] = pd.to_numeric(df[col], downcast="float")
        elif df[col].dtype == "object":
            nunique_ratio = df[col].nunique(dropna=False) / max(len(df), 1)
            if nunique_ratio < 0.5:
                df[col] = df[col].astype("category")
    return df


def smart_read_csv(path: str | Path, nrows: Optional[int] = None) -> pd.DataFrame:
    df = pd.read_csv(path, nrows=nrows, low_memory=False)
    return reduce_memory_usage(df)


def ensure_binary_label(y: pd.Series) -> pd.Series:
    if pd.api.types.is_bool_dtype(y):
        return y.astype(int)
    uniq = pd.Series(y.dropna().unique())
    if set(map(str, sorted(uniq.astype(str)))) <= {"0", "1"}:
        return y.astype(int)
    if len(uniq) == 2:
        vals = list(uniq)
        mapping = {vals[0]: 0, vals[1]: 1}
        return y.map(mapping).astype(int)
    raise ValueError("Label column is not binary. Pass the correct label column.")


def rank_normalize_scores(scores: np.ndarray) -> np.ndarray:
    return pd.Series(scores).rank(method="average", pct=True).values


def add_transaction_features(df: pd.DataFrame, time_col: Optional[str], label_col: Optional[str]) -> pd.DataFrame:
    df = df.copy()
    cmap = lower_map(df.columns)

    amt_col = None
    for c in ["transactionamt", "amount", "amt", "transaction_amount"]:
        if c in cmap:
            amt_col = cmap[c]
            break

    user_col = None
    for c in ["card1", "user", "userid", "customer_id", "account_id"]:
        if c in cmap:
            user_col = cmap[c]
            break

    if time_col and time_col in df.columns:
        if np.issubdtype(df[time_col].dtype, np.number):
            seconds = pd.to_numeric(df[time_col], errors="coerce")
            df["hour_proxy"] = ((seconds // 3600) % 24).astype("float32")
            df["day_proxy"] = ((seconds // (24 * 3600)) % 31).astype("float32")
        else:
            t = pd.to_datetime(df[time_col], errors="coerce")
            df["hour"] = t.dt.hour.astype("float32")
            df["dayofweek"] = t.dt.dayofweek.astype("float32")
            df["day"] = t.dt.day.astype("float32")

    if amt_col and amt_col in df.columns:
        amt = pd.to_numeric(df[amt_col], errors="coerce")
        df[f"{amt_col}_log1p"] = np.log1p(amt.clip(lower=0))
        df[f"{amt_col}_is_large"] = (amt > amt.quantile(0.95)).astype("int8")

    if user_col and amt_col and user_col in df.columns and amt_col in df.columns:
        grp = df.groupby(user_col)[amt_col]
        df[f"{user_col}_{amt_col}_mean"] = grp.transform("mean")
        df[f"{user_col}_{amt_col}_std"] = grp.transform("std")
        df[f"{user_col}_{amt_col}_max"] = grp.transform("max")
        df[f"{amt_col}_to_user_mean_ratio"] = (
            pd.to_numeric(df[amt_col], errors="coerce") /
            (pd.to_numeric(df[f"{user_col}_{amt_col}_mean"], errors="coerce") + 1e-6)
        ).astype("float32")
        temp = df[[user_col]].copy()
        temp["_ones"] = 1
        df["txn_count_per_user"] = temp.groupby(user_col).cumcount().astype("float32")

    return reduce_memory_usage(df)


def build_preprocessor(X: pd.DataFrame):
    numeric_cols = [c for c in X.columns if pd.api.types.is_numeric_dtype(X[c])]
    categorical_cols = [c for c in X.columns if c not in numeric_cols]

    num_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="median")),
        ("scaler", StandardScaler(with_mean=False)),
    ])

    cat_pipe = Pipeline([
        ("imputer", SimpleImputer(strategy="most_frequent")),
        ("encoder", OrdinalEncoder(handle_unknown="use_encoded_value", unknown_value=-1)),
    ])

    preprocessor = ColumnTransformer([
        ("num", num_pipe, numeric_cols),
        ("cat", cat_pipe, categorical_cols),
    ])
    return preprocessor, numeric_cols, categorical_cols


def temporal_split(df: pd.DataFrame, label_col: str, time_col: Optional[str], test_size: float = 0.2):
    df = df.copy()
    if time_col and time_col in df.columns:
        if not np.issubdtype(df[time_col].dtype, np.number):
            df[time_col] = pd.to_datetime(df[time_col], errors="coerce")
        df = df.sort_values(time_col)
        split_idx = int(len(df) * (1 - test_size))
        train_df = df.iloc[:split_idx].copy()
        test_df = df.iloc[split_idx:].copy()
    else:
        train_df, test_df = train_test_split(
            df, test_size=test_size, random_state=42, stratify=df[label_col]
        )
    return train_df, test_df


def summarize_predictions(y_true: np.ndarray, scores: np.ndarray, threshold: float) -> Dict[str, float]:
    y_pred = (scores >= threshold).astype(int)
    tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
    return {
        "threshold": float(threshold),
        "auprc": float(average_precision_score(y_true, scores)),
        "roc_auc": float(roc_auc_score(y_true, scores)) if len(np.unique(y_true)) > 1 else float("nan"),
        "precision": float(precision_score(y_true, y_pred, zero_division=0)),
        "recall": float(recall_score(y_true, y_pred, zero_division=0)),
        "f1": float(f1_score(y_true, y_pred, zero_division=0)),
        "tp": int(tp), "fp": int(fp), "tn": int(tn), "fn": int(fn),
    }


def best_f1_threshold(y_true: np.ndarray, scores: np.ndarray):
    precision, recall, thresholds = precision_recall_curve(y_true, scores)
    f1s = 2 * precision[:-1] * recall[:-1] / np.maximum(precision[:-1] + recall[:-1], 1e-12)
    idx = int(np.nanargmax(f1s))
    th = thresholds[idx]
    return float(th), summarize_predictions(y_true, scores, th)


def cost_optimal_threshold(y_true: np.ndarray, scores: np.ndarray, fn_cost: float = 500.0, fp_cost: float = 2.0):
    thresholds = np.unique(np.quantile(scores, np.linspace(0.001, 0.999, 300)))
    best = None
    rows = []
    n = len(y_true)
    for th in thresholds:
        y_pred = (scores >= th).astype(int)
        tn, fp, fn, tp = confusion_matrix(y_true, y_pred, labels=[0, 1]).ravel()
        total_cost = fn * fn_cost + fp * fp_cost
        row = {
            "threshold": float(th),
            "fp": int(fp),
            "fn": int(fn),
            "tp": int(tp),
            "tn": int(tn),
            "total_cost": float(total_cost),
            "cost_per_transaction": float(total_cost / n),
        }
        rows.append(row)
        if best is None or row["cost_per_transaction"] < best["cost_per_transaction"]:
            best = row
    return best, pd.DataFrame(rows)


def fit_iforest(X_train, contamination):
    model = IsolationForest(
        n_estimators=200, contamination=contamination, random_state=42, n_jobs=-1
    )
    model.fit(X_train)
    return model


def fit_lof(X_train, contamination):
    model = LocalOutlierFactor(
        n_neighbors=35, contamination=contamination, novelty=True, n_jobs=-1
    )
    model.fit(X_train)
    return model


def fit_ocsvm(X_train, contamination):
    nu = min(max(contamination, 0.001), 0.5)
    model = OneClassSVM(kernel="rbf", gamma="scale", nu=nu)
    model.fit(X_train)
    return model


def fit_elliptic(X_train, contamination):
    model = EllipticEnvelope(contamination=contamination, random_state=42)
    model.fit(X_train)
    return model


def fit_gradient_boosting(X_train, y_train):
    pos = max(int((y_train == 1).sum()), 1)
    neg = max(int((y_train == 0).sum()), 1)
    scale_pos_weight = neg / pos

    try:
        from xgboost import XGBClassifier
        model = XGBClassifier(
            n_estimators=300,
            max_depth=6,
            learning_rate=0.05,
            subsample=0.8,
            colsample_bytree=0.8,
            eval_metric="logloss",
            scale_pos_weight=scale_pos_weight,
            random_state=42,
            tree_method="hist",
        )
        model.fit(X_train, y_train)
        return model, "XGBoost"
    except Exception:
        pass

    try:
        from lightgbm import LGBMClassifier
        model = LGBMClassifier(
            n_estimators=300,
            learning_rate=0.05,
            num_leaves=31,
            class_weight={0: 1, 1: scale_pos_weight},
            random_state=42,
        )
        model.fit(X_train, y_train)
        return model, "LightGBM"
    except Exception:
        pass

    sample_weight = np.where(y_train == 1, min(scale_pos_weight, 500.0), 1.0)
    model = HistGradientBoostingClassifier(
        learning_rate=0.05,
        max_depth=8,
        max_iter=250,
        min_samples_leaf=50,
        random_state=42,
    )
    model.fit(X_train, y_train, sample_weight=sample_weight)
    return model, "HistGradientBoosting"


def get_scores(model, X, method_name):
    if method_name in ["IsolationForest", "LOF", "OneClassSVM", "EllipticEnvelope"]:
        return -model.score_samples(X)
    if hasattr(model, "predict_proba"):
        return model.predict_proba(X)[:, 1]
    if hasattr(model, "decision_function"):
        return model.decision_function(X)
    raise ValueError(f"Cannot score model: {method_name}")


def evaluate_model(model, model_name, X_test, y_test, output_dir: Path, dataset_name: str):
    scores = get_scores(model, X_test, model_name)
    if model_name in ["IsolationForest", "LOF", "OneClassSVM", "EllipticEnvelope"]:
        scores = rank_normalize_scores(scores)

    _, f1_metrics = best_f1_threshold(y_test, scores)
    cost_best, cost_df = cost_optimal_threshold(y_test, scores)
    cost_df.to_csv(output_dir / f"{dataset_name}_{model_name.lower()}_cost_curve.csv", index=False)

    return {
        "model": model_name,
        "f1_optimal": f1_metrics,
        "cost_optimal": cost_best,
    }


def run_experiment(
    df: pd.DataFrame,
    label_col: str,
    time_col: Optional[str],
    dataset_name: str,
    output_dir: Path = OUTPUT_DIR,
    max_rows_note: Optional[int] = None,
    run_ocsvm: bool = False,
    run_elliptic: bool = False,
):
    output_dir.mkdir(parents=True, exist_ok=True)

    df = df.copy()
    df[label_col] = ensure_binary_label(df[label_col])

    drop_cols = []
    for c in df.columns:
        if c == label_col:
            continue
        if c.lower().endswith("id") and df[c].nunique(dropna=False) > 0.95 * len(df):
            drop_cols.append(c)
    if drop_cols:
        df = df.drop(columns=drop_cols)

    df = add_transaction_features(df, time_col, label_col)
    train_df, test_df = temporal_split(df, label_col, time_col)

    X_train = train_df.drop(columns=[label_col])
    y_train = train_df[label_col].values
    X_test = test_df.drop(columns=[label_col])
    y_test = test_df[label_col].values

    preprocessor, numeric_cols, categorical_cols = build_preprocessor(X_train)

    X_train_normals = X_train.loc[train_df[label_col] == 0]
    X_train_normals_t = preprocessor.fit_transform(X_train_normals)
    X_train_t = preprocessor.transform(X_train)
    X_test_t = preprocessor.transform(X_test)

    contamination = max(float((y_train == 1).mean()), 0.001)

    results = []

    if_model = fit_iforest(X_train_normals_t, contamination)
    results.append(evaluate_model(if_model, "IsolationForest", X_test_t, y_test, output_dir, dataset_name))

    lof_model = fit_lof(X_train_normals_t, contamination)
    results.append(evaluate_model(lof_model, "LOF", X_test_t, y_test, output_dir, dataset_name))

    if run_ocsvm:
        ocsvm_model = fit_ocsvm(X_train_normals_t, contamination)
        results.append(evaluate_model(ocsvm_model, "OneClassSVM", X_test_t, y_test, output_dir, dataset_name))

    if run_elliptic:
        elliptic_model = fit_elliptic(X_train_normals_t, contamination)
        results.append(evaluate_model(elliptic_model, "EllipticEnvelope", X_test_t, y_test, output_dir, dataset_name))

    gb_model, gb_name = fit_gradient_boosting(X_train_t, y_train)
    results.append(evaluate_model(gb_model, gb_name, X_test_t, y_test, output_dir, dataset_name))

    leaderboard = pd.DataFrame([{
        "model": r["model"],
        "auprc": r["f1_optimal"]["auprc"],
        "roc_auc": r["f1_optimal"]["roc_auc"],
        "precision": r["f1_optimal"]["precision"],
        "recall": r["f1_optimal"]["recall"],
        "f1": r["f1_optimal"]["f1"],
        "f1_threshold": r["f1_optimal"]["threshold"],
        "cost_optimal_threshold": r["cost_optimal"]["threshold"],
        "cost_per_txn": r["cost_optimal"]["cost_per_transaction"],
    } for r in results]).sort_values("auprc", ascending=False)

    leaderboard.to_csv(output_dir / f"{dataset_name}_leaderboard.csv", index=False)

    summary = {
        "dataset_name": dataset_name,
        "label_col": label_col,
        "time_col": time_col,
        "n_rows": int(len(df)),
        "n_train": int(len(train_df)),
        "n_test": int(len(test_df)),
        "fraud_rate_train": float((y_train == 1).mean()),
        "fraud_rate_test": float((y_test == 1).mean()),
        "n_numeric_features": len(numeric_cols),
        "n_categorical_features": len(categorical_cols),
        "models_run": leaderboard["model"].tolist(),
        "max_rows_note": max_rows_note,
    }

    with open(output_dir / f"{dataset_name}_summary.json", "w", encoding="utf-8") as f:
        json.dump(summary, f, indent=2)

    print(f"\nFinished: {dataset_name}")
    print(f"Output folder: {output_dir}")
    print(leaderboard.to_string(index=False))
    return leaderboard


def load_ieee_cis(max_train_rows: Optional[int] = None):
    train_txn = smart_read_csv(IEEE_TRAIN_TRANSACTION, nrows=max_train_rows)
    train_id = smart_read_csv(IEEE_TRAIN_IDENTITY, nrows=max_train_rows)
    train_df = train_txn.merge(train_id, on="TransactionID", how="left")
    return train_df, "isFraud", "TransactionDT"


def load_generic(csv_path: str | Path, max_rows: Optional[int] = None):
    df = smart_read_csv(csv_path, nrows=max_rows)
    label_col = detect_label_column(df)
    time_col = detect_time_column(df)
    return df, label_col, time_col


def inspect_columns(csv_path: str | Path, nrows: int = 5):
    df = pd.read_csv(csv_path, nrows=nrows, low_memory=False)
    print(f"\nFile: {csv_path}")
    print("Columns:")
    print(list(df.columns))
    print("\nPreview:")
    print(df.head())
    return df


In [2]:
print('BASE_DIR =', BASE_DIR)
print('OUTPUT_DIR =', OUTPUT_DIR)

BASE_DIR = C:\Users\alexa\Downloads\kaggle
OUTPUT_DIR = C:\Users\alexa\Downloads\kaggle\project_outputs


In [3]:
ieee_df, ieee_label_col, ieee_time_col = load_ieee_cis(max_train_rows=300000)

ieee_results = run_experiment(
    df=ieee_df,
    label_col=ieee_label_col,
    time_col=ieee_time_col,
    dataset_name='ieee_cis_project',
    output_dir=OUTPUT_DIR,
    max_rows_note=300000,
    run_ocsvm=False,
    run_elliptic=False,
)

ieee_results


Finished: ieee_cis_project
Output folder: C:\Users\alexa\Downloads\kaggle\project_outputs
          model    auprc  roc_auc  precision   recall       f1  f1_threshold  cost_optimal_threshold  cost_per_txn
        XGBoost 0.607018 0.910634   0.791759 0.461803 0.583356      0.882560                0.090926      1.665933
IsolationForest 0.213968 0.797720   0.291667 0.309442 0.300292      0.958817                0.001017      1.920333
            LOF 0.188110 0.742046   0.263900 0.281116 0.272236      0.958650                0.001017      1.920333


,model,auprc,roc_auc,precision,recall,f1,f1_threshold,cost_optimal_threshold,cost_per_txn
2,XGBoost,0.607018,0.910634,0.791759,0.461803,0.583356,0.882560,0.090926,1.665933
0,IsolationForest,0.213968,0.797720,0.291667,0.309442,0.300292,0.958817,0.001017,1.920333
1,LOF,0.188110,0.742046,0.263900,0.281116,0.272236,0.958650,0.001017,1.920333
